<a href="https://colab.research.google.com/github/RoshnaGeorge/ML/blob/main/Lab02_A8_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A8: Data Imputation


In [7]:
import pandas as pd
import numpy as np

# Load the thyroid dataset
thyroid_data = pd.read_excel(
    "Lab Session Data.xlsx",
    sheet_name="thyroid0387_UCI"
)

# Display dataset shape
print("Dataset Shape:", thyroid_data.shape)

Dataset Shape: (9172, 31)


In [8]:
# Replace '?' with NaN
thyroid_data = thyroid_data.mask(thyroid_data == '?', np.nan)

# Display missing values in each attribute
print("Missing Values Before Imputation:")
print(thyroid_data.isnull().sum())

Missing Values Before Imputation:
Record ID                       0
age                             0
sex                           307
on thyroxine                    0
query on thyroxine              0
on antithyroid medication       0
sick                            0
pregnant                        0
thyroid surgery                 0
I131 treatment                  0
query hypothyroid               0
query hyperthyroid              0
lithium                         0
goitre                          0
tumor                           0
hypopituitary                   0
psych                           0
TSH measured                    0
TSH                           842
T3 measured                     0
T3                           2604
TT4 measured                    0
TT4                           442
T4U measured                    0
T4U                           809
FTI measured                    0
FTI                           802
TBG measured                    0
TBG           

identify numeric and categorical attributes

In [9]:
# Try to convert applicable columns to numeric
for column in thyroid_data.columns:
    converted = pd.to_numeric(thyroid_data[column], errors='coerce')

    # Convert to numeric if all non-missing original values are numeric
    if converted.notna().sum() == thyroid_data[column].notna().sum():
        thyroid_data[column] = converted

# Identify numeric and categorical columns
numeric_columns = thyroid_data.select_dtypes(include=[np.number]).columns
categorical_columns = thyroid_data.select_dtypes(exclude=[np.number]).columns

print("Numeric Attributes:")
print(numeric_columns.tolist())

print("\nCategorical Attributes:")
print(categorical_columns.tolist())

Numeric Attributes:
['Record ID', 'age', 'TSH', 'T3', 'TT4', 'T4U', 'FTI', 'TBG']

Categorical Attributes:
['sex', 'on thyroxine', 'query on thyroxine', 'on antithyroid medication', 'sick', 'pregnant', 'thyroid surgery', 'I131 treatment', 'query hypothyroid', 'query hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych', 'TSH measured', 'T3 measured', 'TT4 measured', 'T4U measured', 'FTI measured', 'TBG measured', 'referral source', 'Condition']


numeric column for outliers using IQR

In [10]:
# Impute missing values in numeric attributes

for column in numeric_columns:

    data = thyroid_data[column].dropna()

    # Calculate IQR
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Check for outliers
    outliers = data[(data < lower_bound) | (data > upper_bound)]

    if len(outliers) > 0:
        # Use median if outliers are present
        fill_value = data.median()
        thyroid_data[column] = thyroid_data[column].fillna(fill_value)

        print(column, "-> Median Imputation")

    else:
        # Use mean if no outliers are present
        fill_value = data.mean()
        thyroid_data[column] = thyroid_data[column].fillna(fill_value)

        print(column, "-> Mean Imputation")

Record ID -> Mean Imputation
age -> Median Imputation
TSH -> Median Imputation
T3 -> Median Imputation
TT4 -> Median Imputation
T4U -> Median Imputation
FTI -> Median Imputation
TBG -> Median Imputation


categorical

In [11]:
# Impute missing values in categorical attributes using mode

for column in categorical_columns:

    if thyroid_data[column].isnull().any():
        mode_value = thyroid_data[column].mode()[0]

        thyroid_data[column] = thyroid_data[column].fillna(mode_value)

        print(column, "-> Mode Imputation")

sex -> Mode Imputation


verify missing values

In [12]:
# Check missing values after imputation
print("Missing Values After Imputation:")
print(thyroid_data.isnull().sum())

# Display total number of remaining missing values
print(
    "\nTotal Missing Values:",
    thyroid_data.isnull().sum().sum()
)

Missing Values After Imputation:
Record ID                    0
age                          0
sex                          0
on thyroxine                 0
query on thyroxine           0
on antithyroid medication    0
sick                         0
pregnant                     0
thyroid surgery              0
I131 treatment               0
query hypothyroid            0
query hyperthyroid           0
lithium                      0
goitre                       0
tumor                        0
hypopituitary                0
psych                        0
TSH measured                 0
TSH                          0
T3 measured                  0
T3                           0
TT4 measured                 0
TT4                          0
T4U measured                 0
T4U                          0
FTI measured                 0
FTI                          0
TBG measured                 0
TBG                          0
referral source              0
Condition                    0
dtype: